# GNN Stability Analysis: Baseline Results
## Cora Dataset - Edge Perturbation Experiments

**Author:** Senior Data Science Student  
**Supervisor:** Professor He  
**Date:** June 1, 2026  

This notebook analyzes the baseline GCN stability experiments on the Cora dataset under varying levels of edge perturbation.

In [4]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Experimental Results

In [5]:
import json

with open('../results/local_stability_metrics.json', 'r') as f:
    results = json.load(f)

print(results.keys())

dict_keys(['perturbation_0pct', 'perturbation_5pct', 'perturbation_10pct', 'perturbation_20pct'])


In [6]:
first_key = list(results.keys())[0]
print(first_key)
print(results[first_key].keys())
print(results[first_key])

perturbation_0pct
dict_keys(['mean_accuracy', 'std_deviation', 'raw_scores'])
{'mean_accuracy': 0.797, 'std_deviation': 0.004472135954999583, 'raw_scores': [0.79, 0.798, 0.794, 0.802, 0.801]}


In [3]:
# Extract perturbation levels and metrics
perturbation_levels = [0, 5, 10, 20]
mean_test_accs = []
std_test_accs = []

for p in perturbation_levels:
    key = f'perturbation_{p}pct'
    # Current JSON format uses 'mean_accuracy' and 'std_deviation'
    mean_test_accs.append(results[key]['mean_accuracy'])
    std_test_accs.append(results[key]['std_deviation'])

print("Loaded results for perturbation levels:", perturbation_levels)
print(f"Test accuracies: {[f'{acc*100:.2f}%' for acc in mean_test_accs]}")

## 2. Summary Statistics Table

**Note:** The current experiment only tracks test accuracy. Validation and training accuracy will be added when `local_stability_test.py` is upgraded to track all three splits.

In [ ]:
# Create summary table
print("\n=== Experimental Results Summary ===")
print(f"{'Perturbation (%)':<20} {'Mean Test Acc (%)':<20} {'Std Dev (%)':<15} {'Delta from Baseline (%)':<25}")
print("="*80)
for i, p in enumerate(perturbation_levels):
    mean_pct = mean_test_accs[i] * 100
    std_pct = std_test_accs[i] * 100
    delta_pct = (mean_test_accs[i] - mean_test_accs[0]) * 100
    print(f"{p:<20} {mean_pct:<20.2f} {std_pct:<15.2f} {delta_pct:<25.2f}")

## 3. Perturbation Curve Visualization

In [ ]:
# Create output directory if it doesn't exist
Path('../results/figures').mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 6))

# Convert to percentages
mean_test_pct = np.array(mean_test_accs) * 100
std_test_pct = np.array(std_test_accs) * 100

# Plot test accuracy with error bars
ax.errorbar(perturbation_levels, mean_test_pct, yerr=std_test_pct, 
            marker='o', markersize=8, linewidth=2, capsize=5, capthick=2,
            label='Test Accuracy', color='#2E86AB', alpha=0.9)

# Formatting
ax.set_xlabel('Edge Perturbation Level (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('GCN Stability Under Edge Perturbations (Cora Dataset)', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xticks(perturbation_levels)

# Add baseline reference line
ax.axhline(y=mean_test_pct[0], color='gray', linestyle='--', alpha=0.5, linewidth=1.5, label='Baseline')

plt.tight_layout()
plt.savefig('../results/figures/perturbation_curve.png', dpi=300, bbox_inches='tight')
print("\n✅ Saved figure to: results/figures/perturbation_curve.png")
plt.show()

## 4. Variance Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot standard deviation trends
ax.plot(perturbation_levels, std_test_pct, marker='o', markersize=8, 
        linewidth=2.5, label='Test Std Dev', color='#2E86AB')

ax.set_xlabel('Edge Perturbation Level (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Standard Deviation (%)', fontsize=13, fontweight='bold')
ax.set_title('Variance in GCN Performance Across Seeds', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(perturbation_levels)

plt.tight_layout()
plt.savefig('../results/figures/variance_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Saved figure to: results/figures/variance_analysis.png")
plt.show()

## 5. Individual Seed Performance

In [ ]:
# Extract individual run data from raw_scores
# Note: We don't have seed labels in current format, so we'll just plot the 5 runs
seed_data = []

for p in perturbation_levels:
    key = f'perturbation_{p}pct'
    seed_data.append(results[key]['raw_scores'])

# Transpose to get per-seed trajectories
seed_data = np.array(seed_data).T * 100  # Convert to percentage

# Plot individual seed trajectories
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.viridis(np.linspace(0, 0.9, len(seed_data)))
for idx, trajectory in enumerate(seed_data):
    ax.plot(perturbation_levels, trajectory, marker='o', 
            linewidth=1.5, alpha=0.7, label=f'Run {idx+1}', color=colors[idx])

# Add mean line
ax.plot(perturbation_levels, mean_test_pct, marker='D', markersize=10,
        linewidth=3, color='black', label='Mean', linestyle='--')

ax.set_xlabel('Edge Perturbation Level (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('Individual Run Performance Trajectories', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_xticks(perturbation_levels)

plt.tight_layout()
plt.savefig('../results/figures/seed_trajectories.png', dpi=300, bbox_inches='tight')
print("✅ Saved figure to: results/figures/seed_trajectories.png")
plt.show()

## 6. Key Observations

### Current Experiment Summary

The current experiment shows a **monotonic decrease** in GCN accuracy as edge deletion increases:

- **Baseline (0%):** 79.70% ± 0.45%
- **5% perturbation:** 79.46% ± 0.63% (Δ = -0.24%)
- **10% perturbation:** 78.72% ± 0.92% (Δ = -0.98%)
- **20% perturbation:** 77.94% ± 0.59% (Δ = -1.76%)

### Variance Analysis

The **largest variability** appears at 10% perturbation (std = 0.92%), suggesting an **instability threshold** where the GCN becomes most sensitive to random seed initialization. Variance drops slightly at 20%, possibly due to structural collapse into a more uniform degraded state.

### Run Consistency

All 5 runs show similar degradation patterns with no outliers detected. The consistency across runs suggests the perturbation effect is systematic rather than random.

### Current Limitations

**Note:** Validation and training accuracy will be added later when `local_stability_test.py` is upgraded to track all three data splits. This will allow us to:
- Analyze train/val/test gaps
- Detect overfitting to corrupted structure
- Verify generalization under perturbations

## 7. Next Steps

1. **Upgrade experiment script:** Add validation and training accuracy tracking
2. **Finer granularity:** Add 2.5%, 7.5%, 15% perturbation levels to better characterize the 5-10% transition zone
3. **Spectral analysis:** Compute eigenvalue distributions of perturbed Laplacians
4. **Lipschitz bounds:** Measure empirical Lipschitz constants
5. **Architecture comparison:** Test GAT and other architectures
6. **Regularization:** Implement Lipschitz-constrained variants

## 8. Export Summary for README

In [ ]:
# Generate markdown table for README
print("\n=== Markdown Table for README ===")
print("| Perturbation Level | Mean Test Acc | Std Dev | Delta from Baseline |")
print("|-------------------|---------------|---------|---------------------|")
for i, p in enumerate(perturbation_levels):
    print(f"| {p}% | {mean_test_pct[i]:.2f}% | ±{std_test_pct[i]:.2f}% | {(mean_test_pct[i] - mean_test_pct[0]):.2f}% |")